## Section 15: Deployment Summary and Next Steps

### Project Summary

This notebook has successfully:
1. ✓ Loaded and explored the diabetes dataset
2. ✓ Preprocessed and cleaned the data
3. ✓ Performed comprehensive exploratory data analysis
4. ✓ Scaled features using StandardScaler
5. ✓ Handled class imbalance using SMOTE + Under-sampling
6. ✓ Trained 7 different classification models
7. ✓ Evaluated and compared model performance
8. ✓ Performed hyperparameter tuning
9. ✓ Analyzed feature importance
10. ✓ Saved the best model and scaler for deployment

### Next Steps

1. **Run the Flask Web Application:**
   - Execute `python app.py` to start the Flask server
   - Access the application at `http://localhost:5000`

2. **Train the Model:**
   - Execute `python train.py` to retrain with your own dataset
   - This will generate the best model based on your data

3. **Make Predictions:**
   - Use the web interface to input medical parameters
   - Get real-time diabetes risk predictions
   - Receive personalized recommendations

### Future Enhancements

- **IoT Integration**: Connect with wearable devices for real-time data
- **Biometric Authentication**: Add secure login for patient data
- **Database Integration**: Store predictions and patient history
- **Mobile App**: Develop native mobile applications
- **Deep Learning**: Implement neural networks for enhanced accuracy
- **Multi-language Support**: Extend application language support

In [ ]:
# Test prediction with sample data
print("\n" + "="*60)
print("TEST PREDICTIONS WITH SAMPLE DATA")
print("="*60)

# Sample 1: Likely Non-Diabetic
sample_1 = np.array([[1, 85, 66, 29, 0, 26.6, 0.351, 31]])
sample_1_scaled = scaler.transform(sample_1)
pred_1 = best_model_tuned.predict(sample_1_scaled)[0]
prob_1 = best_model_tuned.predict_proba(sample_1_scaled)[0]

print("\nSample 1 (Likely Non-Diabetic):")
print(f"  Pregnancies: 1, Glucose: 85, BP: 66, SkinThickness: 29")
print(f"  Insulin: 0, BMI: 26.6, Pedigree: 0.351, Age: 31")
print(f"  Prediction: {'Diabetic' if pred_1 == 1 else 'Non-Diabetic'}")
print(f"  Confidence (Non-Diabetic): {prob_1[0]:.4f}")
print(f"  Confidence (Diabetic): {prob_1[1]:.4f}")

# Sample 2: Likely Diabetic
sample_2 = np.array([[6, 148, 72, 35, 0, 33.6, 0.627, 50]])
sample_2_scaled = scaler.transform(sample_2)
pred_2 = best_model_tuned.predict(sample_2_scaled)[0]
prob_2 = best_model_tuned.predict_proba(sample_2_scaled)[0]

print("\nSample 2 (Likely Diabetic):")
print(f"  Pregnancies: 6, Glucose: 148, BP: 72, SkinThickness: 35")
print(f"  Insulin: 0, BMI: 33.6, Pedigree: 0.627, Age: 50")
print(f"  Prediction: {'Diabetic' if pred_2 == 1 else 'Non-Diabetic'}")
print(f"  Confidence (Non-Diabetic): {prob_2[0]:.4f}")
print(f"  Confidence (Diabetic): {prob_2[1]:.4f}")

print("\n✓ Sample predictions completed")

## Section 14: Test Predictions with Sample Data

In [ ]:
# Confusion matrix
y_pred = best_model_tuned.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Non-Diabetic', 'Diabetic'],
            yticklabels=['Non-Diabetic', 'Diabetic'])
axes[0].set_title(f'Confusion Matrix - {best_model_name}', fontweight='bold', fontsize=12)
axes[0].set_ylabel('True Label', fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontweight='bold')

# Classification report
from sklearn.metrics import ConfusionMatrixDisplay
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['Non-Diabetic', 'Diabetic']))

# ROC curve
if hasattr(best_model_tuned, 'predict_proba'):
    y_pred_proba = best_model_tuned.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel('False Positive Rate', fontweight='bold')
    axes[1].set_ylabel('True Positive Rate', fontweight='bold')
    axes[1].set_title(f'ROC Curve - {best_model_name}', fontweight='bold', fontsize=12)
    axes[1].legend(loc="lower right")
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Model analysis completed")

## Section 13: Detailed Model Analysis

In [ ]:
# Save the best model and scaler
import os

model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, 'best_model.pkl')
scaler_path = os.path.join(model_dir, 'scaler.pkl')

# Save model
joblib.dump(best_model_tuned, model_path)
print(f"✓ Best model saved to: {model_path}")

# Save scaler
joblib.dump(scaler, scaler_path)
print(f"✓ Scaler saved to: {scaler_path}")

# Create model info
model_info = {
    'model_name': best_model_name,
    'accuracy': accuracy_score(y_test, best_model_tuned.predict(X_test)),
    'feature_names': X.columns.tolist(),
    'training_samples': len(X_train_balanced),
    'test_samples': len(X_test)
}

print("\n" + "="*60)
print("MODEL SUMMARY")
print("="*60)
for key, value in model_info.items():
    print(f"{key}: {value}")
    
print("\n✓ Model preparation complete!")

## Section 12: Save the Best Model

In [ ]:
# Feature importance for tree-based models
if hasattr(best_model_tuned, 'feature_importances_'):
    feature_importance = best_model_tuned.feature_importances_
    feature_names = X.columns
    
    # Create dataframe for better visualization
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': feature_importance
    }).sort_values('Importance', ascending=False)
    
    print("\n" + "="*60)
    print("FEATURE IMPORTANCE")
    print("="*60)
    print(importance_df.to_string(index=False))
    
    # Visualize feature importance
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(importance_df)))
    ax.barh(importance_df['Feature'], importance_df['Importance'], color=colors)
    
    ax.set_xlabel('Importance Score', fontweight='bold', fontsize=12)
    ax.set_title(f'Feature Importance - {best_model_name}', fontweight='bold', fontsize=14)
    ax.grid(alpha=0.3, axis='x')
    
    # Add value labels
    for i, v in enumerate(importance_df['Importance']):
        ax.text(v + 0.005, i, f'{v:.4f}', va='center')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Top 3 important features:")
    for idx, row in importance_df.head(3).iterrows():
        print(f"  {row['Feature']}: {row['Importance']:.4f}")
    
else:
    print("This model does not have feature importance attribute")

## Section 11: Feature Importance Analysis

In [ ]:
# Find best model for tuning
best_model_name = results_df.loc[results_df['Accuracy'].idxmax(), 'Model']
best_model_initial = trained_models[best_model_name]

print(f"Best Model: {best_model_name}")
print(f"Best Accuracy: {results_df['Accuracy'].max():.4f}")

# Hyperparameter tuning for best model
if best_model_name == 'Random Forest':
    print(f"\nPerforming hyperparameter tuning for {best_model_name}...")
    
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [5, 10, 15],
        'min_samples_split': [2, 5, 10]
    }
    
    grid_search = GridSearchCV(
        RandomForestClassifier(random_state=42, n_jobs=-1),
        param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
    )
    
    grid_search.fit(X_train_balanced, y_train_balanced)
    
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best CV Score: {grid_search.best_score_:.4f}")
    
    # Update best model
    best_model_tuned = grid_search.best_estimator_
    
else:
    print(f"\nUsing {best_model_name} without extensive tuning")
    best_model_tuned = best_model_initial

# Evaluate tuned model
y_pred_best = best_model_tuned.predict(X_test)
best_accuracy = accuracy_score(y_test, y_pred_best)
print(f"\nTuned model test accuracy: {best_accuracy:.4f}")

print("\n✓ Hyperparameter tuning completed")

## Section 10: Hyperparameter Tuning

In [ ]:
# Cross-validation scores
print("\n" + "="*80)
print("CROSS-VALIDATION SCORES (5-Fold)")
print("="*80)

cv_results = []
for name, model in trained_models.items():
    cv_scores = cross_val_score(model, X_train_balanced, y_train_balanced, cv=5, scoring='accuracy')
    cv_results.append({
        'Model': name,
        'Mean CV Accuracy': cv_scores.mean(),
        'Std CV Accuracy': cv_scores.std(),
        'Min CV Accuracy': cv_scores.min(),
        'Max CV Accuracy': cv_scores.max()
    })
    
    print(f"\n{name}:")
    print(f"  Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"  Scores: {[f'{score:.4f}' for score in cv_scores]}")

cv_results_df = pd.DataFrame(cv_results)
print("\n" + "="*80)
print("CROSS-VALIDATION SUMMARY")
print("="*80)
print(cv_results_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = plt.cm.Set3(np.linspace(0, 1, len(results_df)))

for idx, metric in enumerate(metrics):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]
    
    ax.barh(results_df['Model'], results_df[metric], color=colors)
    ax.set_xlabel(metric, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.grid(alpha=0.3, axis='x')
    
    # Add value labels
    for i, v in enumerate(results_df[metric]):
        ax.text(v + 0.02, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

print("✓ Model comparison visualization completed")

In [ ]:
# Evaluate all models on test set
evaluation_results = []

print("\n" + "="*80)
print("MODEL EVALUATION ON TEST SET")
print("="*80)

for name, model in trained_models.items():
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # Calculate ROC-AUC if probability available
    try:
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_pred_proba)
    except:
        roc_auc = None
    
    evaluation_results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    })
    
    print(f"\n{name}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    if roc_auc:
        print(f"  ROC-AUC:   {roc_auc:.4f}")

# Create comparison dataframe
results_df = pd.DataFrame(evaluation_results)
print("\n" + "="*80)
print("COMPARISON TABLE")
print("="*80)
print(results_df.to_string(index=False))

## Section 9: Model Evaluation and Comparison

In [ ]:
# Define multiple classification models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, solver='lbfgs'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5, learning_rate=0.1),
    'SVM': SVC(kernel='rbf', random_state=42, probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
    'Naive Bayes': GaussianNB()
}

print("="*60)
print("TRAINING MULTIPLE MODELS")
print("="*60)

# Train all models
trained_models = {}
for name, model in models.items():
    print(f"\nTraining {name}...", end=" ")
    model.fit(X_train_balanced, y_train_balanced)
    trained_models[name] = model
    print("✓ Complete")

print("\n✓ All models trained successfully!")

## Section 8: Train Multiple Classification Models

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

print("\nTraining set class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest set class distribution:")
print(pd.Series(y_test).value_counts())

# Apply imbalance handling to training data only
X_train_balanced, y_train_balanced = imbalance_pipeline.fit_resample(X_train, y_train)

print(f"\n✓ Data split completed (80% train, 20% test)")
print(f"Training set after balancing: {X_train_balanced.shape[0]} samples")
print("\nBalanced training set class distribution:")
print(pd.Series(y_train_balanced).value_counts())

## Section 7: Train-Test Split

In [ ]:
# Apply SMOTE + Under-sampling to handle class imbalance
print("Original class distribution:")
print(y.value_counts())
print(f"\nClass imbalance ratio: {y.value_counts()[1] / y.value_counts()[0]:.2f}")

# Create pipeline for handling imbalance
imbalance_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42, k_neighbors=5)),
    ('under', RandomUnderSampler(random_state=42))
])

# Apply resampling (only on training data - we'll do this after train-test split)
print("\n✓ Imbalance handling pipeline created (SMOTE + Under-sampling)")
print("  Will be applied to training data after train-test split")

## Section 6: Handling Class Imbalance

In [ ]:
# Separate features and target
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature names:")
print(X.columns.tolist())

# Initialize and fit scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to DataFrame for better visualization
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("\n✓ Features scaled successfully using StandardScaler")
print("\nScaled features statistics:")
print(X_scaled_df.describe())

## Section 5: Feature Engineering and Scaling

In [ ]:
# Box plots to visualize outliers and distributions by outcome
fig, axes = plt.subplots(4, 2, figsize=(14, 12))
fig.suptitle('Feature Distribution by Diabetes Outcome', fontsize=16, fontweight='bold')

for idx, feature in enumerate(features):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]
    
    df_clean.boxplot(column=feature, by='Outcome', ax=ax)
    ax.set_xlabel('Outcome (0: Non-Diabetic, 1: Diabetic)', fontweight='bold')
    ax.set_ylabel(feature, fontweight='bold')
    ax.set_title(f'{feature} by Outcome')
    
plt.suptitle('', fontsize=1)  # Remove automatic title
plt.tight_layout()
plt.show()

print("✓ Box plots generated")

In [ ]:
# Correlation analysis
fig, ax = plt.subplots(figsize=(10, 8))

correlation_matrix = df_clean.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, cbar_kws={'label': 'Correlation'}, ax=ax)

ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print("✓ Correlation matrix generated")
print("\nTop correlations with Outcome:")
correlations_with_outcome = correlation_matrix['Outcome'].sort_values(ascending=False)
print(correlations_with_outcome)

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
outcome_counts = df_clean['Outcome'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Non-Diabetic (0)', 'Diabetic (1)'], outcome_counts.values, color=colors, edgecolor='black')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Variable Distribution (Counts)', fontweight='bold')
axes[0].grid(alpha=0.3, axis='y')

# Percentage plot
axes[1].pie(outcome_counts.values, labels=['Non-Diabetic (0)', 'Diabetic (1)'], 
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Target Variable Distribution (Percentage)', fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Target variable distribution analyzed")

In [ ]:
# Distribution of features
fig, axes = plt.subplots(4, 2, figsize=(14, 12))
fig.suptitle('Distribution of Medical Features', fontsize=16, fontweight='bold')

features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

for idx, feature in enumerate(features):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]
    
    ax.hist(df_clean[feature], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    ax.set_xlabel(feature, fontweight='bold')
    ax.set_ylabel('Frequency')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Feature distributions visualized")

## Section 4: Exploratory Data Analysis (EDA)

In [ ]:
# Handle missing values (fill with mean)
print("Handling Missing Values...")
df_clean = df.copy()

# Fill missing values with mean for numerical columns
numeric_columns = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mean(), inplace=True)

print(f"Missing values after handling: {df_clean.isnull().sum().sum()}")

# Remove duplicates
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
removed_duplicates = initial_rows - len(df_clean)
print(f"Duplicates removed: {removed_duplicates}")

print("\nData Preprocessing Complete!")
print(f"Final dataset shape: {df_clean.shape}")

## Section 3: Data Preprocessing and Cleaning

In [ ]:
# Check for missing values and data types
print("\n" + "="*60)
print("Missing Values:")
print("="*60)
print(df.isnull().sum())

print("\n" + "="*60)
print("Data Types:")
print("="*60)
print(df.dtypes)

print("\n" + "="*60)
print("Target Variable Distribution:")
print("="*60)
print(df['Outcome'].value_counts())
print(f"\nNon-Diabetic: {(df['Outcome'] == 0).sum()} ({(df['Outcome'] == 0).sum() / len(df) * 100:.2f}%)")
print(f"Diabetic: {(df['Outcome'] == 1).sum()} ({(df['Outcome'] == 1).sum() / len(df) * 100:.2f}%)")

In [ ]:
# Load the diabetes dataset
data_path = '../data/diabetes_data.csv'
df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
print("\n" + "="*60)
print("First 5 Rows of the Dataset:")
print("="*60)
print(df.head())

print("\n" + "="*60)
print("Dataset Information:")
print("="*60)
print(df.info())

print("\n" + "="*60)
print("Statistical Summary:")
print("="*60)
print(df.describe())

## Section 2: Load and Explore the Dataset

In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             confusion_matrix, roc_auc_score, classification_report, roc_curve)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
import warnings

warnings.filterwarnings('ignore')

# Configure visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ All libraries imported successfully!")

## Section 1: Import Required Libraries

# Machine Learning-Based Diabetes Prediction System
## Comprehensive Data Exploration and Model Development

This notebook provides a complete walkthrough of developing a machine learning system for diabetes prediction, including:
- Data preprocessing and cleaning
- Exploratory data analysis
- Feature engineering and scaling
- Class imbalance handling
- Multiple model training and evaluation
- Model optimization and deployment

**Project Goals:**
- Build accurate diabetes prediction models
- Identify key medical parameters influencing diabetes
- Deploy model in Flask web application for real-time predictions